# OmniCT — Colab training notebook

Runs the full OmniCT experiment grid (4 methods × 3 seeds + saliency) on **NoduleMNIST3D** (a public 3D CT lung-nodule malignancy dataset, ~50 MB, no auth) in roughly **30–90 minutes** of sequential wall clock on an A100.

**Before you start**:
1. Set runtime to A100 GPU (`Runtime → Change runtime type → A100 GPU`).
2. Make sure your repo is pushed to GitHub. Edit `GITHUB_REPO` below if your URL is different.
3. Run cells top-to-bottom. Each section is idempotent.

**Critical**: do not skip section 5 (data download) or section 5b (pre-flight). If section 5b succeeds you have a working pipeline; if it fails, fix it before running sections 7–10. Earlier failures all came from running the training cells before the data was downloaded.

## 1. Hardware & runtime sanity check

In [ ]:
!nvidia-smi | head -20
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Torch:', torch.__version__)

## 2. Mount Google Drive (for persistent results)

Colab disks are wiped when the runtime ends. We back up `results/`, `report/figures/`, and `data/manifests/` to Drive so a disconnect doesn't lose work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/OmniCT'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive backup root:', DRIVE_ROOT)

## 3. Clone repo & install dependencies

Colab already ships with `torch`, `numpy`, `pandas`, `scikit-learn`, `matplotlib`, `pyyaml`, `tqdm`, `tensorboard`, etc. We only install what's missing.

In [ ]:
GITHUB_REPO = 'https://github.com/pranavraghav75/OmniCT.git'  # ← EDIT if different
BRANCH = 'main'

import os, subprocess
if not os.path.isdir('/content/OmniCT/.git'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, GITHUB_REPO, '/content/OmniCT'], check=True)
else:
    subprocess.run(['git', '-C', '/content/OmniCT', 'pull'], check=True)
%cd /content/OmniCT
!git log -1 --oneline

In [ ]:
%pip install -q "monai>=1.4,<2.0" nibabel pydicom SimpleITK omegaconf einops peft huggingface_hub timm
%pip install -q seaborn medmnist
print('Done installing.')

In [ ]:
import sys
if '/content/OmniCT' not in sys.path:
    sys.path.insert(0, '/content/OmniCT')
import torch, monai, sklearn, numpy, pandas, omegaconf
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('monai', monai.__version__)
print('sklearn', sklearn.__version__, 'numpy', numpy.__version__)

## 4. Smoke test on synthetic data (≤2 minutes)

Confirms training pipeline + GPU work end-to-end before we touch real data.

In [ ]:
!python -m src.training.train \
  --config src/configs/baseline_cnn3d.yaml \
  --override seed=0 device=cuda train.epochs=1 \
    data.synthetic=true data.synthetic_n_train=32 data.synthetic_n_val=8 \
    data.spatial_size=[32,32,32] train.batch_size=4 train.num_workers=2 \
  --run_name colab_smoke_cnn3d

In [ ]:
import json, pathlib
metrics = json.loads(pathlib.Path('results/colab_smoke_cnn3d/metrics.json').read_text())
print(json.dumps(metrics, indent=2))

## 5. Download NoduleMNIST3D (real CT lung-nodule malignancy data, ~50 MB, ~30 sec)

Public, no auth, no quota issues. Same clinical task as the proposal (binary lung-nodule malignancy from CT). The downloader writes everything in OmniCT's standard layout (`data/raw/*.nii.gz`, `data/manifests/labels.csv`, plus per-split id files) so the existing pipeline runs unmodified.

Set `MAX_PER_SPLIT` to limit data per split if you want even faster runs. Leave it as `None` to use all 1633 volumes.

In [ ]:
MAX_PER_SPLIT = None  # set to e.g. 200 for very fast iteration; None = use all
flag = f'--max_per_split {MAX_PER_SPLIT} --balance' if MAX_PER_SPLIT else ''
!python data/download_medmnist.py --out data/raw {flag} --seed 0

# Idempotent manifest fix: strip a leading "raw/" from each path so the
# join with data_root="data/raw" doesn't double up. Safe to run multiple
# times. (Older revisions of the downloader wrote paths relative to
# data/ rather than data/raw/.)
import pandas as pd, pathlib
m_path = pathlib.Path('data/manifests/labels.csv')
df = pd.read_csv(m_path)
before = df['path'].iloc[0]
df['path'] = df['path'].str.replace(r'^raw/', '', regex=True)
after = df['path'].iloc[0]
df.to_csv(m_path, index=False)
print(f'manifest path normalisation: "{before}" -> "{after}"')

# Spot-check that the first volume actually exists at data_root + path.
import os
sample = os.path.join('data/raw', df['path'].iloc[0])
assert os.path.exists(sample), f'expected {sample} to exist after download'
print(f'OK: {sample} ({os.path.getsize(sample)} bytes)')

!ls data/raw | head -5
!echo "---"
!head -5 data/manifests/labels.csv
!wc -l data/manifests/labels.csv data/manifests/splits/*.txt

### Optional: quick manifest fix (no re-download needed)

Run only this cell if you already downloaded data with an older revision of `download_medmnist.py` that wrote paths with a leading `raw/`. After it succeeds, jump straight to pre-flight (section 5b).

In [ ]:
# --- Quick manifest-only fix (run this if you've already downloaded data) ---
# Patches data/manifests/labels.csv in place to strip any leading "raw/"
# from the path column. The data_root config value is already "data/raw/",
# so manifest paths must be just the filename to avoid a doubled "raw/raw/".
# Idempotent: safe to run repeatedly.
import pandas as pd, pathlib, os
m_path = pathlib.Path('data/manifests/labels.csv')
df = pd.read_csv(m_path)
n_changed = int(df['path'].str.startswith('raw/').sum())
df['path'] = df['path'].str.replace(r'^raw/', '', regex=True)
df.to_csv(m_path, index=False)
sample = os.path.join('data/raw', df['path'].iloc[0])
assert os.path.exists(sample), f'expected {sample} to exist'
print(f'normalised {n_changed}/{len(df)} rows')
print(f'first row now resolves to: {sample} ({os.path.getsize(sample)} bytes)')

In [ ]:
# MedMNIST already provides official train/val/test splits (written by
# the downloader above). Skip make_splits. Verify they exist:
!ls -la data/manifests/splits/
print('train:', sum(1 for _ in open('data/manifests/splits/train.txt')))
print('val:  ', sum(1 for _ in open('data/manifests/splits/val.txt')))
print('test: ', sum(1 for _ in open('data/manifests/splits/test.txt')))

## 5b. Pre-flight: one-epoch real-data run (~30 sec)

This must produce a `metrics.json` before you continue. If it errors out, the most common cause is that section 5's download didn't actually populate `data/manifests/labels.csv` — re-run it.

In [ ]:
# Pre-flight: one tiny real-data run that must succeed before the full grid.
# If this cell errors out, do NOT continue to the training cells; fix the data
# state first (e.g., re-run the download cell above).
!python -m src.training.train \
  --config src/configs/baseline_cnn3d.yaml \
  --override seed=0 device=cuda train.epochs=1 \
    data.synthetic=false 'data.spatial_size=[32,32,32]' \
    'data.spacing=[1.0,1.0,1.0]' 'data.hu_window=[-1000.0,400.0]' \
    train.batch_size=8 train.num_workers=2 \
  --run_name preflight_real
import json, pathlib, sys
mp = pathlib.Path('results/preflight_real/metrics.json')
assert mp.exists(), f'preflight failed: {mp} missing -- check the cell output above'
print('preflight OK:', json.loads(mp.read_text()))

## 6. Helper: run a config across N seeds

In [ ]:
import subprocess, time, pathlib, shlex, sys

# Overrides shared across every real-data run.
# - synthetic=false: actually use data/manifests/labels.csv
# - spatial_size=[32,32,32]: matches NoduleMNIST3D's 28^3 native size
#   (small pad/crop, no excessive zero padding)
# - spacing=[1,1,1]: NoduleMNIST volumes have identity affine, so we
#   skip resampling. (For real CT data with metric voxel sizes, use the
#   default 1.5 mm.)
COMMON_OVERRIDES = [
    'data.synthetic=false',
    'data.spatial_size=[32,32,32]',
    'data.spacing=[1.0,1.0,1.0]',
    'data.hu_window=[-1000.0,400.0]',
]

def run_grid(config_path, n_seeds=3, extra_overrides=None):
    extras = list(COMMON_OVERRIDES) + list(extra_overrides or [])
    cfg_stem = pathlib.Path(config_path).stem
    failures = []
    for seed in range(n_seeds):
        run_name = f'{cfg_stem}_seed{seed}'
        cmd = ['python', '-m', 'src.training.train',
               '--config', config_path,
               '--run_name', run_name,
               '--override', f'seed={seed}', 'device=cuda', *extras]
        print('\n>>>', ' '.join(shlex.quote(c) for c in cmd))
        t0 = time.time()
        rc = subprocess.run(cmd).returncode
        elapsed = (time.time() - t0) / 60
        m = pathlib.Path('results') / run_name / 'metrics.json'
        if rc != 0 or not m.exists():
            failures.append(run_name)
            print(f'    -> FAILED in {elapsed:.1f} min  (rc={rc}, metrics={m.exists()})',
                  file=sys.stderr)
        else:
            print(f'    -> ok    in {elapsed:.1f} min')
    if failures:
        raise RuntimeError(f'{len(failures)} run(s) failed: {failures}. '
                           f'Inspect their results/<run>/train.log for the traceback.')

def backup_to_drive():
    """Copy results+figures+manifests to Drive so disconnects are recoverable."""
    !mkdir -p {DRIVE_ROOT}/results {DRIVE_ROOT}/report_figures {DRIVE_ROOT}/manifests
    !rsync -a --delete results/ {DRIVE_ROOT}/results/
    !rsync -a --delete report/figures/ {DRIVE_ROOT}/report_figures/ 2>/dev/null || true
    !rsync -a --delete data/manifests/ {DRIVE_ROOT}/manifests/
    print('Backed up to', DRIVE_ROOT)

## 7. Baseline 1: class-prior (instant)

In [ ]:
run_grid('src/configs/baseline_prior.yaml', n_seeds=3,
         extra_overrides=['train.epochs=1', 'train.batch_size=8', 'train.num_workers=2'])
backup_to_drive()

## 8. Baseline 2: small 3D CNN trained from scratch (~5–15 min total)

In [ ]:
run_grid('src/configs/baseline_cnn3d.yaml', n_seeds=3,
         extra_overrides=['train.epochs=30', 'train.batch_size=16', 'train.num_workers=2'])
backup_to_drive()

## 9. Linear probe on frozen backbone (~5–10 min)

Frozen `random_init` SmallCNN3D feature extractor + a linear head. Edit `src/configs/linear_probe.yaml` if you later want a real foundation model (3DINO/SPECTRE).

In [ ]:
run_grid('src/configs/linear_probe.yaml', n_seeds=3,
         extra_overrides=['train.epochs=25', 'train.batch_size=16', 'train.num_workers=2'])
backup_to_drive()

## 10. LoRA / full fine-tune (~10–20 min total)

The `lora.yaml` config tries to inject LoRA adapters into Linear/attention layers. With the `random_init` SmallCNN3D backbone there are no such layers, so the registry falls back to **end-to-end fine-tuning** (encoder + head) — making this a meaningful contrast with the frozen-backbone linear probe in section 9. The fallback is logged at runtime.

In [ ]:
run_grid('src/configs/lora.yaml', n_seeds=3,
         extra_overrides=['train.epochs=30', 'train.batch_size=16', 'train.num_workers=2'])
backup_to_drive()

## 11. Aggregate results into mean±std table

In [ ]:
!python -m src.training.aggregate_results \
  --runs_dir results --out results/tables/main.csv --split test
import pandas as pd
tbl = pd.read_csv('results/tables/main.csv')
tbl

## 12. Saliency / Grad-CAM panel

In [ ]:
import os
os.makedirs('report/figures', exist_ok=True)
!python -m src.explain.run_saliency \
  --config src/configs/lora.yaml \
  --checkpoint results/lora_seed0/best.pt \
  --override data.synthetic=false device=cuda \
    data.spatial_size=[32,32,32] data.spacing=[1.0,1.0,1.0] \
    data.hu_window=[-1000.0,400.0] \
  --n_samples 6 \
  --methods grad_cam grad_x_input \
  --out report/figures/saliency_panel.pdf

In [ ]:
backup_to_drive()
print('All done. Inspect Drive at:', DRIVE_ROOT)

## Optional: data-efficiency experiment (subsample train set)

Drop in if you have time. Trains LoRA at 25%, 50%, 100% of training data.

In [ ]:
# Quick data-efficiency hack: subsample the train split file in-place,
# train, restore. Only run if you've already finished the main grid.
import shutil, random, pathlib

train_full = pathlib.Path('data/manifests/splits/train.txt')
train_bak  = train_full.with_suffix('.full.txt')
if not train_bak.exists():
    shutil.copy(train_full, train_bak)
full_lines = train_bak.read_text().strip().splitlines()
print('full train size:', len(full_lines))

for frac in [0.25, 0.5, 1.0]:
    for seed in range(3):
        rng = random.Random(seed)
        sub = rng.sample(full_lines, max(2, int(len(full_lines) * frac)))
        train_full.write_text('\n'.join(sub) + '\n')
        run_name = f'lora_frac{int(frac*100)}_seed{seed}'
        !python -m src.training.train \
          --config src/configs/lora.yaml \
          --run_name {run_name} \
          --override seed={seed} device=cuda data.synthetic=false \
            train.epochs=20 train.batch_size=4 train.num_workers=2

shutil.copy(train_bak, train_full)
backup_to_drive()